# Per-Muscle and Overall Average Metrics

## MuscleMap Thigh — Per-Muscle and Overall Average Metrics

Reads every `eval_notebooks/MuscleMap_results_thigh/*.csv`, computes one summary
row per muscle (mean across all scans), then appends an overall average row.

In [1]:
import pathlib
import re
import pandas as pd

RESULTS_DIR = pathlib.Path('../MuscleMap_results_thigh')

_METRIC_PATTERNS = {
    'dice':             re.compile(r'lower_dice', re.I),
    'hausdorff':        re.compile(r'hausdorff', re.I),
    'jaccard':          re.compile(r'jaccard', re.I),
    'volume_similarity':re.compile(r'volume_similarity', re.I),
    'false_negative':   re.compile(r'falseNegative', re.I),
    'false_positive':   re.compile(r'falsePost', re.I),
}

def canonical_metric(col):
    for name, pat in _METRIC_PATTERNS.items():
        if pat.search(col):
            return name
    return None

def muscle_label(stem):
    parts = stem.removeprefix('df_').removesuffix('_thigh').split('_')
    side = parts[0].upper()
    name = '_'.join(p.capitalize() for p in parts[1:])
    return f'{side}_{name}'

In [2]:
csv_files = sorted(RESULTS_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV files:')
for f in csv_files:
    print(' ', f.name)


Found 8 CSV files:
  df_l_gracilis_thigh.csv
  df_l_gracilis_thigh_LONGER.csv
  df_l_sart_thigh.csv
  df_l_sart_thigh_LONGER.csv
  df_r_gracilis_thigh.csv
  df_r_gracilis_thigh_LONGER.csv
  df_r_sart_thigh.csv
  df_r_sart_thigh_LONGER.csv


In [3]:
METRIC_COLS = list(_METRIC_PATTERNS.keys())
summary_rows = []

for csv_path in csv_files:
    df = pd.read_csv(csv_path, index_col=0)

    # build rename map: original col name -> canonical metric name
    rename = {}
    for col in df.columns:
        m = canonical_metric(col)
        if m:
            rename[col] = m

    df = df.rename(columns=rename)

    # keep only recognised metric columns
    metric_cols_present = [c for c in METRIC_COLS if c in df.columns]
    means = df[metric_cols_present].mean()

    row = {'muscle': muscle_label(csv_path.stem)}
    row.update(means.to_dict())
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('muscle')
summary.insert(0, 'codebase', 'musclemap thigh')

# ── Overall average row ───────────────────────────────────────────────────────
overall = summary.select_dtypes(include='number').mean().rename('Overall_Mean')
overall['codebase'] = 'musclemap thigh'
summary = pd.concat([summary, overall.to_frame().T])
summary.index.name = 'muscle'

print(summary.to_string(float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

                                codebase   dice hausdorff jaccard volume_similarity false_negative false_positive
muscle                                                                                                           
L_Gracilis               musclemap thigh 0.7111   16.1447  0.5676           -1.4402         0.2770         0.0002
L_Gracilis_Thigh_Longer  musclemap thigh 0.7111   16.1447  0.5676           -1.4402         0.2770         0.0002
L_Sart                   musclemap thigh 0.7016   35.1036  0.5575            0.4395         0.2279         0.0003
L_Sart_Thigh_Longer      musclemap thigh 0.7016   35.1036  0.5575            0.4395         0.2279         0.0003
R_Gracilis               musclemap thigh 0.7726   13.0651  0.6490           -0.2005         0.2722         0.0001
R_Gracilis_Thigh_Longer  musclemap thigh 0.7726   13.0651  0.6490           -0.2005         0.2722         0.0001
R_Sart                   musclemap thigh 0.7289   28.5071  0.5927           -0.0907     

In [4]:
numeric_cols = summary.select_dtypes(include='number').columns
summary.reset_index().style \
    .format('{:.4f}', subset=numeric_cols) \
    .hide(axis='index')

muscle,codebase,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive
L_Gracilis,musclemap thigh,0.711086,16.144738,0.567603,-1.440162,0.276950,0.000213
L_Gracilis_Thigh_Longer,musclemap thigh,0.711086,16.144738,0.567603,-1.440162,0.276950,0.000213
L_Sart,musclemap thigh,0.701606,35.103649,0.557453,0.439467,0.227949,0.000338
L_Sart_Thigh_Longer,musclemap thigh,0.701606,35.103649,0.557453,0.439467,0.227949,0.000338
R_Gracilis,musclemap thigh,0.772608,13.065069,0.648985,-0.200498,0.272155,0.000106
R_Gracilis_Thigh_Longer,musclemap thigh,0.772608,13.065069,0.648985,-0.200498,0.272155,0.000106
R_Sart,musclemap thigh,0.728916,28.507110,0.592696,-0.090720,0.273982,0.000201
R_Sart_Thigh_Longer,musclemap thigh,0.728916,28.507110,0.592696,-0.090720,0.273982,0.000201
Overall_Mean,musclemap thigh,0.728554,23.205141,0.591684,-0.322978,0.262759,0.000215


In [5]:
# Save to CSV alongside this notebook
out_path = pathlib.Path('musclemap_thigh_avg_metrics.csv')
summary.to_csv(out_path, float_format='%.4f')
print('Saved to', out_path.resolve())


Saved to C:\Projects\dissector\eval_notebooks\paper_avg_results\musclemap_thigh_avg_metrics.csv


## MuscleMap Whole-Body — Per-Muscle and Overall Average Metrics

In [6]:
RESULTS_DIR_WB = pathlib.Path('../MuscleMap_results_WB')

# WB CSVs have two extra metrics not present in the thigh set
_METRIC_PATTERNS_WB = {
    **_METRIC_PATTERNS,
    'binary_cross_entropy': re.compile(r'binary_cross_entropy', re.I),
    'boundary_iou_3d':      re.compile(r'boundary_iou', re.I),
}

def canonical_metric_wb(col):
    for name, pat in _METRIC_PATTERNS_WB.items():
        if pat.search(col):
            return name
    return None

def muscle_label_wb(stem):
    parts = stem.removeprefix('df_').removesuffix('_NEW').split('_')
    side = parts[0].upper()
    name = '_'.join(p.capitalize() for p in parts[1:])
    return f'{side}_{name}'

csv_files_wb = sorted(RESULTS_DIR_WB.glob('*_NEW.csv'))
print(f'Found {len(csv_files_wb)} WB CSV files:')
for f in csv_files_wb:
    print(' ', f.name)

Found 4 WB CSV files:
  df_l_gracilis_NEW.csv
  df_l_sart_NEW.csv
  df_r_gracilis_NEW.csv
  df_r_sart_NEW.csv


In [7]:
METRIC_COLS_WB = list(_METRIC_PATTERNS_WB.keys())
summary_rows_wb = []

for csv_path in csv_files_wb:
    df = pd.read_csv(csv_path, index_col=0)

    rename = {}
    for col in df.columns:
        m = canonical_metric_wb(col)
        if m:
            rename[col] = m

    df = df.rename(columns=rename)

    metric_cols_present = [c for c in METRIC_COLS_WB if c in df.columns]
    means = df[metric_cols_present].mean()

    row = {'muscle': muscle_label_wb(csv_path.stem)}
    row.update(means.to_dict())
    summary_rows_wb.append(row)

summary_wb = pd.DataFrame(summary_rows_wb).set_index('muscle')
summary_wb.insert(0, 'codebase', 'musclemap WB')

overall_wb = summary_wb.select_dtypes(include='number').mean().rename('Overall_Mean')
overall_wb['codebase'] = 'musclemap WB'
summary_wb = pd.concat([summary_wb, overall_wb.to_frame().T])
summary_wb.index.name = 'muscle'

numeric_cols_wb = summary_wb.select_dtypes(include='number').columns
summary_wb.reset_index().style \
    .format('{:.4f}', subset=numeric_cols_wb) \
    .hide(axis='index')

muscle,codebase,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,binary_cross_entropy,boundary_iou_3d
L_Gracilis,musclemap WB,0.792584,13.934614,0.669452,-0.148564,0.232879,0.000098,0.005051,0.555791
L_Sart,musclemap WB,0.812425,18.340754,0.692017,-0.158216,0.228067,0.000127,0.006649,0.569310
R_Gracilis,musclemap WB,0.801321,15.219108,0.685063,-0.190341,0.237705,0.000078,0.004523,0.571786
R_Sart,musclemap WB,0.793742,19.926226,0.673145,-0.258882,0.278036,0.000080,0.006656,0.554554
Overall_Mean,musclemap WB,0.800018,16.855175,0.679919,-0.189001,0.244172,0.000096,0.005720,0.562860


In [8]:
out_path_wb = pathlib.Path('musclemap_wb_avg_metrics.csv')
summary_wb.to_csv(out_path_wb, float_format='%.4f')
print('Saved to', out_path_wb.resolve())

Saved to C:\Projects\dissector\eval_notebooks\paper_avg_results\musclemap_wb_avg_metrics.csv


## DaFne Thigh — Per-Muscle and Overall Average Metrics

In [9]:
RESULTS_DIR_DAFNE = pathlib.Path('../dafne_thigh_results/results')

# DaFne CSVs add inter-slice dice on top of the WB metric set
_METRIC_PATTERNS_DAFNE = {
    **_METRIC_PATTERNS_WB,
    'inter_slice_dice_pred': re.compile(r'inter_slice_dice_pred', re.I),
    'inter_slice_dice_gt':   re.compile(r'inter_slice_dice_gt',   re.I),
}

def canonical_metric_dafne(col):
    for name, pat in _METRIC_PATTERNS_DAFNE.items():
        if pat.search(col):
            return name
    return None

def muscle_label_dafne(stem):
    parts = stem.removeprefix('df_').removesuffix('_dafne').split('_')
    side = parts[0].upper()
    name = '_'.join(p.capitalize() for p in parts[1:])
    return f'{side}_{name}'

csv_files_dafne = sorted(RESULTS_DIR_DAFNE.glob('*_dafne.csv'))
print(f'Found {len(csv_files_dafne)} DaFne CSV files:')
for f in csv_files_dafne:
    print(' ', f.name)

Found 4 DaFne CSV files:
  df_l_gracilis_dafne.csv
  df_l_sart_dafne.csv
  df_r_gracilis_dafne.csv
  df_r_sart_dafne.csv


In [10]:
METRIC_COLS_DAFNE = list(_METRIC_PATTERNS_DAFNE.keys())
summary_rows_dafne = []

for csv_path in csv_files_dafne:
    df = pd.read_csv(csv_path, index_col=0)

    rename = {}
    for col in df.columns:
        m = canonical_metric_dafne(col)
        if m:
            rename[col] = m

    df = df.rename(columns=rename)

    metric_cols_present = [c for c in METRIC_COLS_DAFNE if c in df.columns]
    means = df[metric_cols_present].mean()

    row = {'muscle': muscle_label_dafne(csv_path.stem)}
    row.update(means.to_dict())
    summary_rows_dafne.append(row)

summary_dafne = pd.DataFrame(summary_rows_dafne).set_index('muscle')
summary_dafne.insert(0, 'codebase', 'dafne thigh')

overall_dafne = summary_dafne.select_dtypes(include='number').mean().rename('Overall_Mean')
overall_dafne['codebase'] = 'dafne thigh'
summary_dafne = pd.concat([summary_dafne, overall_dafne.to_frame().T])
summary_dafne.index.name = 'muscle'

numeric_cols_dafne = summary_dafne.select_dtypes(include='number').columns
summary_dafne.reset_index().style \
    .format('{:.4f}', subset=numeric_cols_dafne) \
    .hide(axis='index')

C:\Users\docto\miniconda3\envs\ultraseg\Lib\site-packages\numpy\_core\_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


muscle,codebase,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,binary_cross_entropy,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt
L_Gracilis,dafne thigh,0.130032,199.444222,0.072550,-1.178387,0.914293,0.000397,0.083822,0.048833,0.590791,0.829984
L_Sart,dafne thigh,0.008839,188.357551,0.004738,0.920438,inf,0.001013,0.023046,0.006559,0.238260,0.867787
R_Gracilis,dafne thigh,0.122674,198.192956,0.068452,-1.115925,inf,0.000443,0.080585,0.048724,0.562623,0.822644
R_Sart,dafne thigh,0.010229,184.785384,0.005525,0.845367,inf,0.000932,0.022062,0.009752,0.243425,0.840573
Overall_Mean,dafne thigh,0.067943,192.695028,0.037816,-0.132127,inf,0.000696,0.052379,0.028467,0.408775,0.840247


In [11]:
out_path_dafne = pathlib.Path('dafne_thigh_avg_metrics.csv')
summary_dafne.to_csv(out_path_dafne, float_format='%.4f')
print('Saved to', out_path_dafne.resolve())

Saved to C:\Projects\dissector\eval_notebooks\paper_avg_results\dafne_thigh_avg_metrics.csv


## Combined Overall Means

In [12]:
overall_combined = pd.concat([
    summary.loc[['Overall_Mean']],
    summary_wb.loc[['Overall_Mean']],
    summary_dafne.loc[['Overall_Mean']],
])
overall_combined.index.name = 'muscle'

numeric_cols_combined = overall_combined.select_dtypes(include='number').columns
overall_combined.reset_index().style \
    .format('{:.4f}', subset=numeric_cols_combined) \
    .hide(axis='index')

muscle,codebase,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,binary_cross_entropy,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt
Overall_Mean,musclemap thigh,0.728554,23.205141,0.591684,-0.322978,0.262759,0.000215,nan,nan,nan,nan
Overall_Mean,musclemap WB,0.800018,16.855175,0.679919,-0.189001,0.244172,0.000096,0.005720,0.562860,nan,nan
Overall_Mean,dafne thigh,0.067943,192.695028,0.037816,-0.132127,inf,0.000696,0.052379,0.028467,0.408775,0.840247


In [ ]:
out_path_combined = pathlib.Path('overall_means.csv')
overall_combined.to_csv(out_path_combined, float_format='%.4f')
print('Saved to', out_path_combined.resolve())